# Roman Urdu Sentiment Analyzer
TF-IDF + LogisticRegression on ~33.6k Roman Urdu texts (Positive / Negative / Neutral).
Data: `ayeshasameer/roman-urdu-sentiment-analysis` (Hugging Face, MIT).

In [ ]:
!pip install -q datasets scikit-learn pandas joblib pyarrow

## 1. Load data

In [ ]:
from datasets import load_dataset
ds = load_dataset('ayeshasameer/roman-urdu-sentiment-analysis')
for s in ds:
    print(s, ds[s].num_rows)

## 2. Clean text

In [ ]:
import re, pandas as pd

def clean(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def frame(split):
    df = ds[split].to_pandas()
    df['text'] = df['text'].map(clean)
    return df[df['text'].str.len() > 0].reset_index(drop=True)

train_df, val_df, test_df = frame('train'), frame('validation'), frame('test')
print(train_df['label'].value_counts())
print(train_df['text'].str.len().describe()[['min','50%','mean','max']].round(1))

## 3. TF-IDF + models

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score

labels = sorted(train_df['label'].unique())
l2i = {l: i for i, l in enumerate(labels)}
for df in (train_df, val_df, test_df):
    df['y'] = df['label'].map(l2i)

vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(train_df['text'])
Xva, Xte = vec.transform(val_df['text']), vec.transform(test_df['text'])
ytr, yva, yte = train_df['y'].values, val_df['y'].values, test_df['y'].values

## 4. Tune C on validation, evaluate on test

In [ ]:
best, best_c, best_clf = -1, None, None
for c in (0.25, 0.5, 1.0, 2.0, 4.0):
    clf = LogisticRegression(C=c, max_iter=2000).fit(Xtr, ytr)
    f1 = f1_score(yva, clf.predict(Xva), average='macro')
    print(f'C={c}: val macro-F1={f1:.4f}')
    if f1 > best:
        best, best_c, best_clf = f1, c, clf
print('best C:', best_c)

## 5. Test results: LR vs baselines

In [ ]:
import pandas as pd
models = {
    'majority': DummyClassifier(strategy='most_frequent'),
    'logreg': best_clf,
    'multinomial_nb': MultinomialNB(),
    'linear_svc': LinearSVC(),
}
rows = []
for name, clf in models.items():
    clf.fit(Xtr, ytr)
    p = clf.predict(Xte)
    rows.append({'model': name,
               'macro_f1': round(f1_score(yte, p, average='macro'), 4),
               'accuracy': round(accuracy_score(yte, p), 4)})
pd.DataFrame(rows).sort_values('macro_f1', ascending=False)

## 6. Save artifacts

In [ ]:
import joblib, json
joblib.dump(vec, 'tfidf_vectorizer.joblib')
joblib.dump(models['logreg'], 'model.joblib')
json.dump(l2i, open('labels.json', 'w'))
print('saved')

## 7. Try it

In [ ]:
for s in ['yeh drama bohat acha hai', 'bekaar service, paisay zaya ho gaye', 'kal office jana hai']:
    pr = models['logreg'].predict_proba(vec.transform([clean(s)]))[0]
    print(f'{s!r} -> {labels[int(pr.argmax())]} ({pr.max():.1%})')